<a href="https://colab.research.google.com/github/Laura-Oliveira-Braga/Aurora-Siger/blob/main/Relat%C3%B3rio_Aurora_Siger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###**Organização e descrição da telemetria**

In [1]:
# Manipulação de dados
import pandas as pd

In [8]:
# Criação e organização do dataset
dados = pd.read_csv('/content/dados_Aurora_Siger.csv', sep=',')

In [10]:
#Primeiros registros
dados.head(10)

,timestamp,internal_temp_C,external_temp_C,structural_integrity,energy_level_pct,tank_pressure_kPa,critical_modules_status
0,2026-03-31T00:00:00Z,22.5,-45.2,1,100,4500,COMMS:OK|NAV:OK|LIFE:OK|PROP:OK
1,2026-03-31T00:10:00Z,22.4,-46.0,1,99,4498,COMMS:OK|NAV:OK|LIFE:OK|PROP:OK
2,2026-03-31T00:20:00Z,22.3,-47.5,1,98,4495,COMMS:OK|NAV:OK|LIFE:OK|PROP:OK
3,2026-03-31T00:30:00Z,22.1,-50.0,1,97,4490,COMMS:OK|NAV:OK|LIFE:OK|PROP:OK
4,2026-03-31T00:40:00Z,21.9,-55.3,1,96,4480,COMMS:OK|NAV:OK|LIFE:OK|PROP:OK
5,2026-03-31T00:50:00Z,21.7,-60.1,1,95,4470,COMMS:OK|NAV:OK|LIFE:OK|PROP:OK
6,2026-03-31T01:00:00Z,21.5,-65.0,1,94,4460,COMMS:OK|NAV:OK|LIFE:OK|PROP:OK
7,2026-03-31T01:10:00Z,21.3,-70.2,1,93,4450,COMMS:OK|NAV:OK|LIFE:OK|PROP:OK
8,2026-03-31T01:20:00Z,21.0,-75.8,1,92,4440,COMMS:OK|NAV:OK|LIFE:OK|PROP:OK
9,2026-03-31T01:30:00Z,20.8,-80.0,1,91,4430,COMMS:OK|NAV:OK|LIFE:OK|PROP:OK


In [12]:
# Quantidade de linhas e colunas
dados.shape

(28, 7)

In [15]:
# Tipos de colunas e valores faltantes
dados.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   timestamp                28 non-null     object 
 1   internal_temp_C          28 non-null     float64
 2   external_temp_C          28 non-null     float64
 3   structural_integrity     28 non-null     int64  
 4   energy_level_pct         28 non-null     int64  
 5   tank_pressure_kPa        28 non-null     int64  
 6   critical_modules_status  28 non-null     object 
dtypes: float64(2), int64(3), object(2)
memory usage: 1.7+ KB


In [16]:
# Resumo numérico
dados.describe()

,internal_temp_C,external_temp_C,structural_integrity,energy_level_pct,tank_pressure_kPa
count,28.000000,28.000000,28.000000,28.000000,28.000000
mean,18.503571,-103.235714,0.750000,85.964286,4220.464286
std,3.228287,40.037558,0.440959,9.106295,299.227584
min,12.500000,-170.000000,0.000000,68.000000,3500.000000
25%,15.875000,-136.250000,0.750000,79.750000,4037.500000
50%,19.250000,-102.500000,1.000000,86.500000,4365.000000
75%,21.350000,-68.900000,1.000000,93.250000,4452.500000
max,22.500000,-45.200000,1.000000,100.000000,4500.000000


###**Algoritmo de verificação**

In [17]:
import csv

In [21]:
# Definição da função
def verificar_decolagem(linha):
  internal_temp = float(linha['internal_temp_C'])
  external_temp = float(linha['external_temp_C'])
  integrity = int(linha['structural_integrity'])
  energy = float(linha['energy_level_pct'])
  pressure = float(linha['tank_pressure_kPa'])
  modules = linha['critical_modules_status'].split('|')

  # Faixas seguras
  if not (15 <= internal_temp <= 25):
    return "DECOLAGEM ABORTADA"
  if not (-150 <= external_temp <= 120):
    return "DECOLAGEM ABORTADA"
  if integrity != 1:
    return "DECOLAGEM ABORTADA"
  if energy < 80:
    return "DECOLAGEM ABORTADA"
  if not (3000 <= pressure <= 5000):
    return "DECOLAGEM ABORTADA"
  if any("OK" not in m for m in modules):
    return "DECOLAGEM ABORTADA"

  return "PRONTO PARA DECOLAR"

###**Script em Python**

In [22]:
# Simulação de dados
dados_simulados = [
    {
        'timestamp': '2026-03-31T00:00:00Z',
        'internal_temp_C': '22.5',
        'external_temp_C': '-45.2',
        'structural_integrity': '1',
        'energy_level_pct': '100',
        'tank_pressure_kPa': '4500',
        'critical_modules_status': 'COMMS:OK|NAV:OK|LIFE:OK|PROP:OK'
    },
    {
        'timestamp': '2026-03-31T03:30:00Z',
        'internal_temp_C': '15.5',
        'external_temp_C': '-140.0',
        'structural_integrity': '0',
        'energy_level_pct': '79',
        'tank_pressure_kPa': '4000',
        'critical_modules_status': 'COMMS:DEGRADED|NAV:OK|LIFE:OK|PROP:DEGRADED'
    }
]

# Execução das verifiações
for linha in dados_simulados:
  resultado = verificar_decolagem(linha)
  print(linha['timestamp'], ":", resultado)

2026-03-31T00:00:00Z : PRONTO PARA DECOLAR
2026-03-31T03:30:00Z : DECOLAGEM ABORTADA


###**Análise energética**

In [24]:
# Definição da função
def calcular_autonomia(capacidade_total, carga_pct, consumo_decolagem, perdas):
   # Energia disponível
   energia_disponível = capacidade_total * (carga_pct / 100)

   # Energia líquida após perdas
   energia_liquida = energia_disponível - perdas

   # Autonomia inicial
   autonomia = energia_liquida - consumo_decolagem

   return autonomia

# Exemplo
capacidade_total = 10000  # KWh
carga_pct = 90            # %
consumo_decolagem = 1500  # KWh
perdas = 200              #KWh

autonomia_inicial = calcular_autonomia(capacidade_total, carga_pct, consumo_decolagem, perdas)
print("Autonomia incial após decolagem:", autonomia_inicial, "KWh")

Autonomia incial após decolagem: 7300.0 KWh


###**Análise assistida por IA**

In [25]:
import pandas as pd

# 1. Classificação de todos os dados
dados['status_decolagem'] = dados.apply(verificar_decolagem, axis=1)

# 2. Identificação de Anomalias
anomalias_integridade = dados[dados['structural_integrity'] == 0]
anomalias_energia = dados[dados['energy_level_pct'] < 80]

print("--- RELATÓRIO DE ANÁLISE ---")
print(f"Total de registros: {len(dados)}")
print(f"Prontos para decolar: {sum(dados['status_decolagem'] == 'PRONTO PARA DECOLAR')}")
print(f"Abortados: {sum(dados['status_decolagem'] == 'DECOLAGEM ABORTADA')}")

print("\n--- ANOMALIAS DETECTADAS ---")
print(f"Falhas de Integridade Estrutural: {len(anomalias_integridade)} instâncias")
print(f"Nível de Energia Crítico (<80%): {len(anomalias_energia)} instâncias")

print("\n--- SUGESTÕES DE RISCO ---")
if len(anomalias_integridade) > 0:
    print("RISCO ESTRUTURAL: Detectada perda de integridade em múltiplos pontos. Verifique fadiga de material.")
if dados['external_temp_C'].min() < -150:
    print("RISCO TÉRMICO: Temperaturas externas atingiram o limite crítico de -150°C. Verifique isolamento.")

display(dados[['timestamp', 'energy_level_pct', 'structural_integrity', 'status_decolagem']].tail(10))

--- RELATÓRIO DE ANÁLISE ---
Total de registros: 28
Prontos para decolar: 15
Abortados: 13

--- ANOMALIAS DETECTADAS ---
Falhas de Integridade Estrutural: 7 instâncias
Nível de Energia Crítico (<80%): 7 instâncias

--- SUGESTÕES DE RISCO ---
RISCO ESTRUTURAL: Detectada perda de integridade em múltiplos pontos. Verifique fadiga de material.
RISCO TÉRMICO: Temperaturas externas atingiram o limite crítico de -150°C. Verifique isolamento.


,timestamp,energy_level_pct,structural_integrity,status_decolagem
18,2026-03-31T03:00:00Z,82,1,DECOLAGEM ABORTADA
19,2026-03-31T03:10:00Z,81,1,DECOLAGEM ABORTADA
20,2026-03-31T03:20:00Z,80,1,DECOLAGEM ABORTADA
21,2026-03-31T03:30:00Z,79,0,DECOLAGEM ABORTADA
22,2026-03-31T03:40:00Z,78,0,DECOLAGEM ABORTADA
23,2026-03-31T03:50:00Z,76,0,DECOLAGEM ABORTADA
24,2026-03-31T04:00:00Z,74,0,DECOLAGEM ABORTADA
25,2026-03-31T04:10:00Z,72,0,DECOLAGEM ABORTADA
26,2026-03-31T04:20:00Z,70,0,DECOLAGEM ABORTADA
27,2026-03-31T04:30:00Z,68,0,DECOLAGEM ABORTADA
